##### __Skill Level:__ <span style="color:blue">Intermediate</span>

# Creating a Chatbot for Structured Data using BigQuery to implement RAG

## Overview

This script builds an autonomous, natural-language data analytics chatbot using Retrieval-Augmented Generation (RAG). By integrating Gemini 2.5 Pro with BigQuery via LangChain, it eliminates the need to manually write complex database code.

Users can ask questions about structured databases in plain English (e.g., "How many female smokers are in the dataset?"). The AI engine dynamically inspects the database schema, structures a flawless SQL query, runs it against your live tables, and translates the raw row results back into a clear text summary.

## Prerequisites

Ensure you have set up your environment and verified the following access requirements:

BigQuery Permissions: Your active Google Cloud service profile requires standard permissions to create and interact with datasets and tables (specifically bigquery.datasets.create, bigquery.tables.create, and bigquery.jobs.create).

## Learning objectives

In this tutorial you will learn about:
- Connect AI to BigQuery: Build a live SQLAlchemy translation layer that links LangChain directly to structured tables.
- Implement Text-to-SQL RAG: Deploy Gemini 2.5 Pro as an autonomous agent that reads schemas, writes valid SQL queries, and summarizes data in plain English.
- Master Modern SDKs: Use current, verified class libraries (ChatGoogleGenerativeAI) and execution methods (.invoke()).How to set up a BigQuery dataset and a table.

## Get Started

### Install Packages

In [ ]:
!pip install --upgrade -q google-genai google-cloud-bigquery pandas pandas-gbq
!pip install google-cloud-bigquery pyarrow pandas

In [ ]:
!pip install pandas pandas-gbq

###  Initialize Environment Constrants & Authentication

In [ ]:
import subprocess
from google import genai
from google.genai import types
from google.cloud import bigquery

# Automatically grab your active cloud lab workspace parameters
PROJECT_ID = subprocess.check_output("gcloud config get-value project", shell=True).decode("utf-8").strip()
LOCATION = "global"

DATASET_NAME = "healthcare_insights"
TABLE_NAME = "screening"

print("Connected to Project: "+ PROJECT_ID)

In [ ]:
# Initialize the official GenAI Client matching your successful script structure

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

# Initialize native BigQuery database compiler
bq_client = bigquery.Client(project=PROJECT_ID)


### Create the Dataset Container (Database) and Load CSV and Upload to BigQuery

In [ ]:
import pandas as pd 

dataset_ref = f"{PROJECT_ID}.{DATASET_NAME}"  
table_id = f"{dataset_ref}.{TABLE_NAME}" 

#bq_client.get_dataset(dataset_ref)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION

# create dataset
bq_client.create_dataset(dataset, timeout=30)
print("Dataset created successfully.")

#Load the CSV data and push to the specific table
df = pd.read_csv("ds_salaries_2026.csv")
job = bq_client.load_table_from_dataframe(df, table_id)
job.result()  # Waits for upload to finish


### DEFINE THE NATIVE SQL TOOL

This function gives Gemini the ability to execute read-only queries against your database table.

In [ ]:
import pandas as pd
from google.api_core.exceptions import GoogleAPICallError

def run_bigquery_query(sql_query: str) -> pd.DataFrame:
    """Executes a SQL query against BigQuery and returns a structured DataFrame.
    
    Raises:
        GoogleAPICallError: If the BigQuery job fails due to syntax or server issues.
    """
    query_job = bq_client.query(sql_query)
    df = query_job.to_dataframe()
    return df

### Native Gemini 2.5 PRO Agent Invocation Engine

This system instructs Gemini 2.5 Pro to leverage the BigQuery function to answer human queries.

Define the query compiler pipeline function

In [ ]:
import json
from google.api_core.exceptions import GoogleAPICallError
from google.genai import types
from google import genai

def ask_gemini_about_database(human_prompt: str, target_sql: str) -> str:
    """Executes a SQL query, safe-parses the data to JSON, and prompts Gemini 3.5 Flash to analyze it."""
    try:
        # 1. Execute query and convert safely to a pandas DataFrame to handle memory efficiently
        query_job = bq_client.query(target_sql)
        df = query_job.to_dataframe()
        
        # Check if table is empty before calling Gemini to save token costs
        if df.empty:
            raw_database_data = "No records matching this query were found in the database."
        else:
            # Limit rows sent to LLM context to prevent token overflow
            sample_df = df.head(50) 
            raw_database_data = sample_df.to_json(orient="records", date_format="iso")
            
    except GoogleAPICallError as db_err:
        error_msg = "Database data extraction error: ".format(db_err.message)
        print(error_msg)
        raise

    # 2. Structure the grounding context using explicit string methods (No f-strings)
    system_instruction = (
        "You are an expert data analyst helper.\n"
        "You are given a question alongside clean JSON entries pulled directly from the BigQuery tables.\n"
        "Analyze the provided data to formulate a clean, short, plain English response that directly answers the user's question."
    )
    
    # Constructing prompt content without f-strings using .format()
    full_content = "User Question: {}\n\nRaw BigQuery Data (JSON):\n{}".format(
        human_prompt, 
        raw_database_data
    )
    
    config = types.GenerateContentConfig(
        system_instruction=system_instruction, 
        temperature=0.2
    )

    # 3. Invoke the specific model target against the global endpoint configuration
    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=full_content,
            config=config
        )
        return response.text
    except Exception as ai_err:
        print("AI generation failed: {}".format(str(ai_err)))
        raise


### Execution Testing Environment

In [ ]:
print("--- TEST 1: Verification Row Check ---")

q1_prompt = "How many total rows of data are inside our data table?"
q1_sql = "SELECT COUNT(*) as total_rows FROM `" + PROJECT_ID + "." + DATASET_NAME + "." + TABLE_NAME + "`"

# Fetching the answer from your refactored function
model_answer = ask_gemini_about_database(q1_prompt, q1_sql)

# Printing using formatting and string multiplication for the divider
print(" Model Answer:\n{}\n{}".format(model_answer, "=" * 60))


In [ ]:
# Prompt 2: Targeted Segment Count Check
print("\n--- TEST 2 ---")

q2_prompt = "How many records represent Data Engineer jobs?"

# Target SQL query built safely using standard string concatenation
q2_sql = (
    "SELECT COUNT(*) as engineer_count FROM `" 
    + PROJECT_ID + "." + DATASET_NAME + "." + TABLE_NAME 
    + "` WHERE job_title = 'Data Engineer'"
)

# Fetching the answer from your automated Gemini analysis pipeline
ans_2 = ask_gemini_about_database(q2_prompt, q2_sql)

# Printing using traditional .format() and string multiplication for the divider
print("Final Answer:\n{}\n{}".format(ans_2, "=" * 60))

In [ ]:
print("\n--- TEST 3 ---")
q3_prompt = "What is the highest salary recorded in USD within this dataset?"
q3_sql = (
    "SELECT MAX(salary_in_usd) as max_salary FROM `" 
    + PROJECT_ID + "." + DATASET_NAME + "." + TABLE_NAME + "`"
)
ans_3 = ask_gemini_about_database(q3_prompt, q3_sql)
print("Final Answer:\n{}\n{}".format(ans_3, "=" * 60))

In [ ]:
print("\n--- TEST 4 ---")
q4_prompt = "How many records represent Senior-level (SE) employees?"
q4_sql = (
    "SELECT COUNT(*) as senior_count FROM `" 
    + PROJECT_ID + "." + DATASET_NAME + "." + TABLE_NAME 
    + "` WHERE experience_level = 'SE'"
)
ans_4 = ask_gemini_about_database(q4_prompt, q4_sql)
print("Final Answer:\n{}\n{}".format(ans_4, "=" * 60))


### Configure Modern Langchain Integration Using Gemini 2.5 Pro

This step constructs the translation layer (SQLAlchemy URI) and hooks up the advanced Gemini 2.5 Pro engine using key authentication tokens.

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Establish the database translator URL mapping LangChain to your project's dataset
sqlalchemy_url = f'bigquery://{PROJECT_ID}/{DATASET_NAME}'
db = SQLDatabase.from_uri(sqlalchemy_url)

# 2. Build the LLM Configuration with Gemini 2.5 Pro
# Using ChatGoogleGenerativeAI ensures it targets the AI Studio key pipeline flawlessly
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",          # Swapped to Google's advanced 2.5 reasoning architecture
    temperature=0,                  # Zero temperature forces accurate, deterministic SQL syntax generation
    max_tokens=8192,
    max_retries=2,
    google_api_key=os.environ["GOOGLE_API_KEY"] # Safely paths via AI Studio to avoid 403 errors
)

print(" Gemini 2.5 Pro & Database mappings bound successfully.")



### Instantiate The Reasoning SQL RAG Agent 

This step sets up the autonomous Agent Executor, giving the AI model tool access to run schemas and build analytics inside your dataset.

In [ ]:
%pip install langchain langchain-community langchain-openai

In [ ]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.utilities.sql_database import SQLDatabase
print(" Assembling the Autonomous Data Analytics Agent...")

# Bundle database schemas and the reasoning engine into a cohesive toolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Instantiate the active framework executor. 
# It converts conversational statements to valid SQL commands behind the scenes.
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,      # Displays the exact code steps the AI is thinking through live
    top_k=100          # Protects your system by capping output rows
)

print(" Agent operational! Unleashing natural language queries:\n")

### Execute Experimentation Queries

This section contains your test prompts. The syntax uses '.invoke()' to stay fully current with modern APIs.

In [ ]:
### Query 1: Total Baseline Rows Verification
response_1 = agent_executor.invoke({"input": "How many rows are in the table screening?"})
print("\nModel Answer 1:\n" + str(response_1['output']) + "\n" + "="*50) 

### Query 2: Conditional Filtering Verification
response_2 = agent_executor.invoke({"input": "How many obese records are inside the table " + TABLE_NAME + "?"})
print("\nModel Answer 2:\n" + str(response_2['output']) + "\n" + "="*50) 

### Query 3: Multi-Conditional Logic Parsing
complex_prompt = (
"How many smokers are female inside the table {}? ".format(TABLE_NAME) +
"Use logic where a value of 1 in the smoke column denotes a smoker, "
"and a value of 2 in the gender column denotes a female."
)
response_3 = agent_executor.invoke({"input": complex_prompt})
print("\nModel Answer 3:\n" + str(response_3['output']) + "\n" + "="*50)